This notebook shows how the lower top_X mids files can be derived from the top_50 filenames 
and the HDF5 file containing the data.
Since the lower numbers are subsets of top_50, the filenames can be filtered and matching 
labels need to be added. The log-melspectrograms are all already contained.

In [ ]:
import os

import h5py
import torch
import numpy as np

import utils.utility_funcs as my_funcs

In [ ]:
PATH_TO_VALID_FILENAMES_DIR = r"C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\valid_filenames"
PATH_TO_LABEL_DICT_DIR = r"C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\labels"

In [ ]:
fn_files = os.scandir(PATH_TO_VALID_FILENAMES_DIR)

valid_filenames = {}
for fn_file in fn_files:
    split_fname = fn_file.name.split('_')
    dataset, datasplit, nr_classes = split_fname[0], split_fname[1], split_fname[3][3:5]
    print(f"dataset: {dataset}, datasplit: {datasplit}, nr_of_classes: {nr_classes}")
    key = f"{dataset}_{datasplit}_{nr_classes}"
    val = my_funcs.read_in_valid_filenames(fn_file.path)
    valid_filenames[key] = val



In [ ]:
label_files = os.scandir(PATH_TO_LABEL_DICT_DIR)

label_dicts = {}
for l_file in label_files:
    split_fname = l_file.name.split('_')
    dataset, datasplit, nr_classes = split_fname[0], split_fname[1], split_fname[3]
    print(f"dataset: {dataset}, datasplit: {datasplit}, nr_of_classes: {nr_classes}")
    key = f"{dataset}_{datasplit}_{nr_classes}"
    val = my_funcs.pickle_load(l_file.path)
    label_dicts[key] = val



In [ ]:
for data_id in valid_filenames.keys():
    filenames = valid_filenames[data_id]
    labels = label_dicts[data_id]


In [ ]:
# Paths to the pckl dicts and the valid filenames for 30 mids
PATH_TO_AS_EVAL_30_LABELS = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\audioset_eval_labels_30_dict.pckl'
PATH_TO_AS_TRAIN_30_LABELS = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\audioset_train_labels_30_dict.pckl'
PATH_TO_FSD50K_EVAL_30_LABELS = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\fsd50k_eval_labels_30_dict.pckl'
PATH_TO_FSD50K_TRAIN_30_LABELS = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\pickle_objects\fsd50k_train_labels_30_dict.pckl'

PATH_TO_AS_FNAMES_EVAL_30 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\audioset_eval_filenames_top30.txt'
PATH_TO_AS_FNAMES_TRAIN_30 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\audioset_train_filenames_top30.txt'
PATH_TO_FSD50K_FNAMES_EVAL_30 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\fsd50k_eval_filenames_top30.txt'
PATH_TO_FSD50K_FNAMES_TRAIN_30 = r'C:\Users\mp431591\Documents\work_code\cl_30\continual_learning\fsd50k_train_filenames_top30.txt'


In [ ]:

PATH_TO_DATA_HDF5 = r'C:\Users\mp431591\Documents\work_code\cl_30\data_cl_backup.hdf5'

In [ ]:
# Read in desired filenames
valid_filenames_as_train_30 = my_funcs.read_in_valid_filenames(PATH_TO_AS_FNAMES_TRAIN_30)
valid_filenames_as_eval_30 = my_funcs.read_in_valid_filenames(PATH_TO_AS_FNAMES_EVAL_30)

valid_filenames_fsd50k_train_30 = my_funcs.read_in_valid_filenames(PATH_TO_FSD50K_FNAMES_TRAIN_30)
valid_filenames_fsd50k_eval_30 = my_funcs.read_in_valid_filenames(PATH_TO_FSD50K_FNAMES_EVAL_30)

valid_filenames_fsd50k_eval_30

In [ ]:
# Get dicts
audioset_train_labels_30 = my_funcs.pickle_load(PATH_TO_AS_TRAIN_30_LABELS)
audioset_eval_labels_30 = my_funcs.pickle_load(PATH_TO_AS_EVAL_30_LABELS)

fsd50k_train_labels_30 = my_funcs.pickle_load(PATH_TO_FSD50K_TRAIN_30_LABELS)
fsd50k_eval_labels_30 = my_funcs.pickle_load(PATH_TO_FSD50K_EVAL_30_LABELS)

fsd50k_eval_labels_30

In [ ]:

# For Audioset. Some differences in filenames between audiofiles and metafiles

with h5py.File(PATH_TO_DATA_HDF5, 'a') as data_hdf5:
    as_train_50_grp = data_hdf5['audioset_strong_train_50']
    as_eval_50_grp = data_hdf5['audioset_strong_eval_50']
    as_grps = [as_train_50_grp, as_eval_50_grp]

    fsd50k_train_50_grp = data_hdf5['fsd50k_train_50']
    fsd50k_eval_50_grp = data_hdf5['fsd50k_eval_50']

    print(as_train_50_grp)
    print(as_eval_50_grp)
    print(fsd50k_train_50_grp)
    print(fsd50k_eval_50_grp)
    for as_grp in as_grps:
        
        split_type = as_grp.name.split('_')[-2]
        print(split_type)
        valid_fnames = ""
        labels = ""

        if split_type == 'train':
            valid_fnames = valid_filenames_as_train_30
            labels = audioset_train_labels_30
        else: #eval
            valid_fnames = valid_filenames_as_eval_30
            labels = audioset_eval_labels_30

        for fname in as_grp.keys():

            fname_wo_Y = fname[1::] # Remove leading Y
            nr_of_classes = str(30) 
            membership_attr = 'in_' + nr_of_classes
            label_attr = 'label_' + nr_of_classes
            
            if fname in valid_fnames:
                # Add two attributes: if the dataset is a part of lower classes
                # and the matching label

                as_grp[fname].attrs[membership_attr] = 1 # True
                as_grp[fname].attrs[label_attr] = labels[fname_wo_Y]
                
            else:
                as_grp[fname].attrs[membership_attr] = 0 # False


# Read through the desired group and the dataset names inside
# Compare names to the valid filename list and copy to a new group
# without label info and insert appropriate label info

In [ ]:
# The same as above but for FSD50k. FSD50k has varying length audiofiles which were processed into 10 sec chunks leading to filenames in the hdf5 being slightly different (an additional '_x', where x: {0, 1, 2}, suffix to identify it) when compared to valid filenames.

with h5py.File(PATH_TO_DATA_HDF5, 'a') as data_hdf5:

    fsd50k_train_50_grp = data_hdf5['fsd50k_train_50']
    fsd50k_eval_50_grp = data_hdf5['fsd50k_eval_50']
    fsd50k_grps = [fsd50k_train_50_grp, fsd50k_eval_50_grp]

    print(fsd50k_train_50_grp)
    print(fsd50k_eval_50_grp)

    for fsd50k_grp in fsd50k_grps:
        
        split_type = fsd50k_grp.name.split('_')[-2]
        valid_fnames = ""
        labels = ""

        if split_type == 'train':
            valid_fnames = valid_filenames_fsd50k_train_30
            labels = fsd50k_train_labels_30
        else: #eval
            valid_fnames = valid_filenames_fsd50k_eval_30
            labels = fsd50k_eval_labels_30

        for fname in fsd50k_grp.keys():

            fname_og = ""
            if '_' in fname:
                fname_og = fname.split('_')[0]
            else:
                fname_og = fname

            nr_of_classes = str(30) 
            membership_attr = 'in_' + nr_of_classes
            label_attr = 'label_' + nr_of_classes
            
            if fname_og in valid_fnames:
                # Add two attributes: if the dataset is a part of lower classes
                # and the matching label

                fsd50k_grp[fname].attrs[membership_attr] = 1 # True
                fsd50k_grp[fname].attrs[label_attr] = labels[fname_og]
                
            else:
                fsd50k_grp[fname].attrs[membership_attr] = 0 # False

In [64]:
counter = 0
with h5py.File(PATH_TO_DATA_HDF5, 'r') as test:
    for grp in test.keys():
        counter = 0
        print(grp)
        tmp_grp = test[grp]
        print(tmp_grp)
        for file in tmp_grp.keys():
            counter += tmp_grp[file].attrs['in_30']
        print(f"in_30 for {tmp_grp}: {counter}")
    
    

audioset_eval_50
<HDF5 group "/audioset_eval_50" (14242 members)>
in_30 for <HDF5 group "/audioset_eval_50" (14242 members)>: 13964
audioset_train_50
<HDF5 group "/audioset_train_50" (81070 members)>
in_30 for <HDF5 group "/audioset_train_50" (81070 members)>: 79689
fsd50k_eval_50
<HDF5 group "/fsd50k_eval_50" (11185 members)>
in_30 for <HDF5 group "/fsd50k_eval_50" (11185 members)>: 10103
fsd50k_train_50
<HDF5 group "/fsd50k_train_50" (37686 members)>
in_30 for <HDF5 group "/fsd50k_train_50" (37686 members)>: 33113


In [ ]:
print('Y--12UOziMF0' in valid_filenames_as_train_30)
print('Y--24LG2mr-Y' in valid_filenames_as_train_30)
print('--12UOziMf0' in audioset_train_labels_30)
print('--24LG2mr-Y' in audioset_train_labels_30)